In [96]:
import numpy as np
import opt_einsum as oe
import matplotlib.pyplot as plt
import scipy as sp

from quant_rotor.CCC_integration_methods.Dense.de_solve_one_thermal import integration_scheme

from quant_rotor.Hamiltonian_models.Dense.operators import rotor_operators, heisenberg_operators
from quant_rotor.Hamiltonian_models.Dense.hamiltonian import hamiltonian_dense

from quant_rotor.Hamiltonian_models.Dense.basis_transform import combine_transform, energy_transform, TF_transform

from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_periodic import t_periodic

from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_exact import t_1_amplitude_guess_ground_state, t_2_amplitude_guess_ground_state, amplitute_energy, intermediate_normalisation

from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_sub_class import (
    QuantumSimulation,
    SimulationParams,
    TensorData,
)
from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_sub_class_linked import (
    QuantumSimulation_linked,
    SimulationParams_linked,
    TensorData_linked,
)

from quant_rotor.CCC_integration_methods.Dense.stat_mech_thermo import U as U_stat

In [2]:
np.set_printoptions(precision=8)
np.set_printoptions(suppress=True)
np.set_printoptions(linewidth=np.inf)
np.set_printoptions(threshold=np.inf)

# CCC Iterative

## Initialization 

In [3]:
n_comb_sites = 2
site_comb = 2
site = n_comb_sites ** site_comb

state = 3
state_comb = state**n_comb_sites
state_TF = state ** 2

g = 0.1

periodic = False

In [4]:
K, V = rotor_operators(state, g)
H = hamiltonian_dense(site_comb, K, V, V, periodic)

eig_val, eig_vec = np.linalg.eigh(H)

In [ ]:
# K_e, V_e, _ = energy_transform(K, V, V)
# K_e_TF, V_e_TF, _ = TF_transform(K, V, V)

In [ ]:
# H_e_TF = hamiltonian_dense(site_comb, K_e_TF, V_e_TF, V_e_TF, periodic)

# eig_val_TF, eig_vec_TF = np.linalg.eigh(H_e_TF)

In [9]:
K_comb, V_comb_xy, V_comb_yx = combine_transform(2, K, V)
H_comb = hamiltonian_dense(site_comb, K_comb, V_comb_xy, V_comb_yx, periodic)

eig_val_comb, eig_vec_comb = np.linalg.eigh(H_comb)

In [10]:
K_e_comb, V_e_comb_xy, V_e_comb_yx = energy_transform(K_comb, V_comb_xy, V_comb_yx)

V_e_comb_xy = V_e_comb_xy.reshape(state_comb, state_comb, state_comb, state_comb)
V_e_comb_yx = V_e_comb_yx.reshape(state_comb, state_comb, state_comb, state_comb)

# for i in range(state_comb):
#     K_e_comb[i, i] = K_e_comb[i, i] - K_e_comb[0, 0]
#     for j in range(state_comb):
#         V_e_comb_xy[i, j, i, j] = V_e_comb_xy[i, j, i, j] - V_e_comb_xy[0, 0, 0, 0]
#         V_e_comb_yx[i, j, i, j] = V_e_comb_yx[i, j, i, j] - V_e_comb_yx[0, 0, 0, 0]

In [11]:
H_e_comb = hamiltonian_dense(site_comb, K_e_comb, V_e_comb_xy.reshape(state_comb**2, state_comb**2), V_e_comb_yx.reshape(state_comb**2, state_comb**2), periodic)

eig_val_e_comb, eig_vec_e_comb = np.linalg.eigh(H_e_comb)

In [53]:
print(eig_val)
# print(eig_val_TF)
print(eig_val_comb)
print(eig_val_e_comb)

[-0.00623059  0.9         0.95        1.05        1.1         2.          2.          2.          2.00623059]
[-0.01871957  0.826175    0.91012439  0.93223084  0.95943633  1.02128658  1.05515109  1.0717262   1.14896987  1.77668144  1.82484481  1.82484481  1.88676906  1.89936305  1.90780543  1.90780543  1.94752693  1.99236788  1.99484802  1.99484802  1.99563308  1.99704835  1.99891854  1.99891854  2.00621434  2.04748551  2.08725922  2.08725922  2.09938773  2.11023711  2.16759366  2.16759366  2.22254409  2.84896179  2.8591057   2.85929902  2.87192236  2.87928932  2.88695012  2.88726209  2.92192236  2.92200465  2.94257991  2.95685845  2.95695103  2.96731801  2.96800667  2.9709467   2.97928932  3.02071068  3.03149085  3.03669921  3.03700393  3.04414505  3.04418701  3.06568524  3.07807764  3.08343738  3.11555877  3.11576129  3.12071068  3.12807764  3.15097722  3.15113716  3.17257246  4.          4.          4.          4.          4.          4.00088883  4.00109572  4.00109572  4.00504716  

In [54]:
t_1_exact = t_1_amplitude_guess_ground_state(state_comb, site_comb, eig_vec_e_comb, eig_val_e_comb)
t_2_exact = t_2_amplitude_guess_ground_state(state_comb, site_comb, eig_vec_e_comb, eig_val_e_comb)

In [55]:
# T_1, T_2, R_1, R_2 = t_periodic(site_comb, state_comb, K_e_comb, V_e_comb_xy, V_e_comb_yx, "original", Import_t=True, t_1_import=np.copy(t_1_new), t_2_import=np.copy(t_2_new), periodic=periodic, one_cicle=True)

In [56]:
E = amplitute_energy(site_comb, state_comb, periodic, K_e_comb, V_e_comb_xy, V_e_comb_yx, np.copy(t_1_exact), np.copy(t_2_exact))

In [57]:
E - eig_val_e_comb[0]

np.complex128(0j)

In [58]:
K_e_copy = np.copy(K_e_comb)
V_e_copy_xy = np.copy(V_e_comb_xy)
V_e_copy_yx = np.copy(V_e_comb_yx)

In [59]:
K_e_comb = K_e_copy
V_e_comb_xy = V_e_copy_xy
V_e_comb_yx = V_e_copy_yx

In [60]:
t_1_new = np.copy(t_1_exact)
t_2_new = np.copy(t_2_exact)

## CI Version 1

In [61]:
p = state_comb
i = 1
a = p - i

epsilon = np.diag(K_e_comb)

In [62]:
params = SimulationParams(
    a=a,
    i=i,
    p=p,  # These can be the same as `a + i` or chosen independently
    site=site_comb,
    state=state_comb,
    i_method=3,
    gap=False,
    gap_site=3,
    epsilon=epsilon,
    periodic=periodic,
)

tensors = TensorData(
    t_a_i_tensor=t_1_new,
    t_ab_ij_tensor=t_2_new,
    h_full=K_e_comb,
    v_full_xy=V_e_comb_xy.reshape(state_comb, state_comb, state_comb, state_comb),
    v_full_yx=V_e_comb_yx.reshape(state_comb, state_comb, state_comb, state_comb),
)

qs = QuantumSimulation(params, tensors)

In [63]:
energy_1 = 0
energy_2 = 0

for site_x in range(site_comb):
    energy_1 += np.einsum("ip, pi->", qs.h_term(i, p), qs.B_term(i, site_x))

    for site_y in range(site_comb):
        if site_x < site_y:
            # if abs(site_x - site_y) == 1:
            # noinspection SpellCheckingInspection
            energy_2 += np.einsum(
                "ijab, abij->",
                qs.v_term(i, i, a, a, site_x, site_y),
                qs.t_term_2(site_x, site_y),
            )
            # noinspection SpellCheckingInspection
            energy_2 += np.einsum(
                "ijpq, pi, qj->",
                qs.v_term(i, i, p, p, site_x, site_y),
                qs.B_term(i, site_x),
                qs.B_term(i, site_y),
            )

In [64]:
C_1 = np.einsum(
    "pi, qj->pqij",
    qs.B_term(i, 0),
    qs.B_term(i, 1))

In [65]:
C_1[1:, 1:] += t_2_new[0, 1]

In [66]:
L_h = np.einsum(
    "pP, Pqij->pqij",
    qs.h_term(p, p),
    C_1)

L_h += np.einsum(
    "pQij, qQ->pqij",
    C_1,
    qs.h_term(p, p))

L_h -= energy_1 * C_1

In [67]:
L_V = np.einsum("pqPQ, PQij -> pqij", qs.v_term(p, p, p, p, 0, 1), C_1) - energy_2 * C_1

In [68]:
L_h.reshape(9, 9)

array([[ 0.        +0.j,  0.        +0.j, -0.        +0.j, -0.        +0.j, -0.        +0.j,  0.        +0.j, -0.00016767+0.j, -0.        +0.j, -0.00012539+0.j],
       [ 0.        +0.j,  0.05475889+0.j, -0.        +0.j, -0.        +0.j, -0.04956588+0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j, -0.        +0.j],
       [ 0.        +0.j, -0.        +0.j, -0.0260925 +0.j,  0.02482967+0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j,  0.        +0.j],
       [-0.        +0.j,  0.        +0.j, -0.02482967+0.j,  0.02362806+0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j,  0.        +0.j],
       [ 0.        +0.j,  0.04956588+0.j,  0.        +0.j, -0.        +0.j, -0.0448645 +0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j],
       [ 0.        +0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j, -0.00001861+0.j,  0.        +0.j, -0.00001619+0.j, -0.        +0.j],
       [-0.00016767+0.j,  0.

In [69]:
L_V.reshape(9, 9)

array([[ 0.        +0.j,  0.        +0.j,  0.        +0.j,  0.        +0.j,  0.        +0.j, -0.        +0.j,  0.00016767+0.j,  0.        +0.j,  0.00012539+0.j],
       [-0.        +0.j, -0.05475889+0.j,  0.        +0.j,  0.        +0.j,  0.04956588+0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j],
       [-0.        +0.j,  0.        +0.j,  0.0260925 +0.j, -0.02482967+0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j, -0.        +0.j],
       [ 0.        +0.j, -0.        +0.j,  0.02482967+0.j, -0.02362806+0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j, -0.        +0.j],
       [ 0.        +0.j, -0.04956588+0.j, -0.        +0.j,  0.        +0.j,  0.0448645 +0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j],
       [-0.        +0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j,  0.00001861+0.j, -0.        +0.j,  0.00001619+0.j,  0.        +0.j],
       [ 0.00016767+0.j, -0.

## CI Version 2

In [70]:
p = state_comb
i = 1
a = p - i

epsilon = np.diag(K_e_comb)
params_linked = SimulationParams_linked(
    a=a,
    i=i,
    p=p,  # These can be the same as `a + i` or chosen independently
    site=site_comb,
    state=state_comb,
    i_method=3,
    gap=False,
    gap_site=3,
    epsilon=epsilon,
    periodic=periodic,
)

tensors_linked = TensorData_linked(
    t_a_i_tensor=t_1_new,
    t_ab_ij_tensor=t_2_new,
    h_full=K_e_comb,
    v_full_xy=V_e_comb_xy.reshape(state_comb, state_comb, state_comb, state_comb),
    v_full_yx=V_e_comb_yx.reshape(state_comb, state_comb, state_comb, state_comb),
)

qs_linked = QuantumSimulation_linked(params_linked, tensors_linked)

In [71]:
def R_1_CI_func(x: int):

    y = (x + 1) % 2

    r_1 = np.einsum("ap, pi -> ai", qs_linked.h_term(a, p), qs_linked.B_term(i, x))

    r_1 += np.einsum("jb, bj, ai -> ai", qs_linked.h_term(i,a), qs_linked.t_term_1(y), qs_linked.t_term_1(x))
    r_1 += np.einsum("ajcd, cdij-> ai", qs_linked.v_term(a, i, a, a, x, y), qs_linked.t_term_2(x, y))
    r_1 += np.einsum("ajpq, pi, qj -> ai", qs_linked.v_term(a, i, p, p, x, y), qs_linked.B_term(i, x), qs_linked.B_term(i, y))
    r_1 -= (qs_linked.ec1(x) + qs_linked.ec1(y) + qs_linked.ec2(x, y)) * tensors_linked.t_a_i_tensor[x]

    return r_1

def R_2_CI_func(x: int, y: int):

    r_2 = np.einsum("ap, pi, bj -> abij", qs_linked.h_term(a, p), qs_linked.B_term(i, x), qs_linked.t_term_1(y))

    r_2 += np.einsum("ac, cbij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(x, y))
    r_2 += np.einsum("bq, ai, qj -> abij", qs_linked.h_term(a, p), qs_linked.t_term_1(x), qs_linked.B_term(i, y))
    r_2 += np.einsum("bd, adij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(x, y))
    r_2 += np.einsum("abpq, pi, qj -> abij", qs_linked.v_term(a, a, p, p, x, y), qs_linked.B_term(i, x), qs_linked.B_term(i, y))
    r_2 += np.einsum("abcd, cdij -> abij", qs_linked.v_term(a, a, a, a, x, y), qs_linked.t_term_2(x, y))
    r_2 -= (np.einsum("ai, bj -> abij", qs_linked.t_term_1(x), qs_linked.t_term_1(y)) + tensors_linked.t_ab_ij_tensor[x, y])*(qs_linked.ec1(x) + qs_linked.ec1(y) + qs_linked.ec2(x, y))

    return r_2

In [72]:
x = 0
y = 1

In [73]:
term_2 = np.einsum("bd, adij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(x, y)).reshape(a, a)

In [74]:
term_1 = np.einsum("ac, cbij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(x, y)).reshape(a, a)

In [75]:
R_10 = np.einsum("ap, pc, cbij->abij",qs.A_term(a, 1),qs.h_term(p, a),qs.t_term_2(1, 0)).reshape(a, a)
R_01 = np.einsum("ap, pc, cbij->abij",qs.A_term(a, 0),qs.h_term(p, a),qs.t_term_2(0, 1)).reshape(a, a)

In [76]:
(t_2_exact[0, 1].reshape(a, a) - t_2_exact[1, 0].reshape(a, a).T)

array([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]])

In [77]:
R_01 - term_1

array([[ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j, -0.+0.j, -0.+0.j, -0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j, -0.+0.j, -0.+0.j, -0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j]])

In [78]:
R_10.T - term_2

array([[ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j, -0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j]])

In [79]:
R_1_CI = np.zeros((site_comb, a, i), dtype=complex)
R_2_CI = np.zeros((site_comb, site, a, a, i, i), dtype=complex)

In [80]:
for x in range(site_comb):
    R_1_CI[x] =  R_1_CI_func(x)
    for y in range(site_comb):
        if x < y:
            R_2_CI[x, y] = R_2_CI_func(x, y)
            R_2_CI[y, x] = R_2_CI[x, y].reshape(a, a).T.reshape(a, a, i, i)

## CCC

In [81]:
T_1, T_2, R_1, R_2 = t_periodic(site_comb, state_comb, K_e_comb, V_e_comb_xy.reshape(state_comb, state_comb, state_comb, state_comb), V_e_comb_yx.reshape(state_comb, state_comb, state_comb, state_comb), "original", Import_t=True, t_1_import=np.copy(t_1_new), t_2_import=np.copy(t_2_new), periodic=periodic, one_cicle=True)

original


In [82]:
T_1_linked, T_2_linked, R_1_linked, R_2_linked = t_periodic(site_comb, state_comb, K_e_comb, V_e_comb_xy.reshape(state_comb, state_comb, state_comb, state_comb), V_e_comb_yx.reshape(state_comb, state_comb, state_comb, state_comb), "linked", Import_t=True, t_1_import=np.copy(t_1_new), t_2_import=np.copy(t_2_new), periodic=periodic, one_cicle=True)

linked


In [83]:
T_1_transformed, T_2_transformed, R_1_transformed, R_2_transformed = t_periodic(site_comb, state_comb, K_e_comb, V_e_comb_xy.reshape(state_comb, state_comb, state_comb, state_comb), V_e_comb_yx.reshape(state_comb, state_comb, state_comb, state_comb), "transformed", Import_t=True, t_1_import=np.copy(t_1_new), t_2_import=np.copy(t_2_new), periodic=periodic, one_cicle=True)

transformed


In [84]:
R_E = L_h + L_V
R_E_x = R_E[1:, 0, 0, 0].reshape(a, i)
R_E_y = R_E[0, 1:, 0, 0].reshape(a, i)
R_E_xy = R_E[1:, 1:, 0, 0].reshape(a, a, i, i)

In [85]:
np.max(np.abs(R_E))

np.float64(2.0816681711721685e-17)

In [86]:
R_E_xy_cor = R_E_xy - np.einsum("ai, bj -> abij", R_E_x, t_1_new[1]) - np.einsum("ai, bj -> abij", R_E_y, t_1_new[0])

## Output

In [87]:
A_h = np.einsum("ap, pc ->ac", qs.A_term(a, 1), qs.h_term(p, a))

In [88]:
R = np.einsum("ap, pc, cbij->abij",qs.A_term(a, 0),qs.h_term(p, a),qs.t_term_2(0, 1),)

In [89]:
r_2 = np.einsum("ac, cbij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(0, 1))

In [90]:
np.max(np.abs(R - r_2))

np.float64(1.2327179617010851e-26)

In [91]:
print("CI site x:", np.max(np.abs(R_E_x - R_1_CI[0])))
print("CI site y:", np.max(np.abs(R_E_y - R_1_CI[1])))
print("CI site xy:", np.max(np.abs(R_E_xy - R_2_CI[0, 1])))

CI site x: 5.20727124949386e-07
CI site y: 5.20727124949114e-07
CI site xy: 0.00037648274570231655


In [95]:
np.max(np.abs(R_1))
np.max(np.abs(R_2))

np.float64(2.0816681711721685e-17)

In [92]:
print("Connected/Linked R_1:" ,np.max(np.abs(R_1_linked - R_1)))
print("Connected/Linked R_2:" ,np.max(np.abs(R_2_linked - R_2)), "\n")

print("Connected/Transformed R_1:" ,np.max(np.abs(R_1_transformed - R_1)))
print("Connected/Transformed R_2:" ,np.max(np.abs(R_2_transformed - R_2)), "\n")

print("Linked/Transformed R_1:" ,np.max(np.abs(R_1_transformed - R_1_linked)))
print("Linked/Transformed R_2:" ,np.max(np.abs(R_2_transformed - R_2_linked)))

Connected/Linked R_1: 5.207271249494944e-07
Connected/Linked R_2: 0.00037648274570230983 

Connected/Transformed R_1: 7.589415207398531e-19
Connected/Transformed R_2: 0.00037648274570230983 

Linked/Transformed R_1: 5.20727124949386e-07
Linked/Transformed R_2: 2.515687853345272e-19


In [93]:
print("R_E site x:" ,np.max(np.abs(R_1[0] - R_1_CI[0])))
print("R_E site y:" ,np.max(np.abs(R_1[0] - R_1_CI[0])))
print("R_E site xy:" ,np.max(np.abs(R_2[0, 1].reshape(a, a) - R_2_CI[0, 1].reshape(a, a))))
print("R_E site xy corrected:" ,np.max(np.abs(R_2[0, 1].reshape(a, a) - R_2_CI[0, 1].reshape(a, a))))

R_E site x: 5.207271249494944e-07
R_E site y: 5.207271249494944e-07
R_E site xy: 0.0003764827457023096
R_E site xy corrected: 0.0003764827457023096
